# VecDB National Parks Quickstart
Experience the oracle-vecdb Python SDK with the National Parks sample data set—connect, explore, search, and manage your vectors in just a few cells.

### What you'll do
1. Environment setup
2. Configure connection
3. Create a client
4. Quick health check
5. Create or reuse the parks table
6. Capture a table overview
7. Browse sample parks
8. Perform a filtered semantic search
9. Generate demo embedding and upsert it
10. Cleanup by deleting the demo embedding

## 1. Environment Setup
- Oracle Autonomous AI Vector Database with the **PARKS** vector table and `all_MiniLM_L12_v2` embedding model loaded into the database.
- `.env` file populated with `VECDB_REST_URL`, `VECDB_USERNAME`, `VECDB_PASSWORD` (copy and amend the .env.example).
- Python 3.10+ virtual environment activated.


In [ ]:
%pip install -U oracle-vecdb python-dotenv pandas

## 2. Configure Connection
Load environment variables and confirm all required hosts, users, and passwords are set before connecting.

In [ ]:
from dotenv import load_dotenv 

load_dotenv()
PARK_TABLE = "PARKS"
N_RESULTS = 5


## 3. Create Client
Instantiate `OracleVecDB` using the loaded configuration for reuse throughout the notebook.

In [ ]:
from dotenv import load_dotenv 

load_dotenv()

import os
from oracle_vecdb import OracleVecDB, Configuration

rest_url = os.getenv("VECDB_REST_URL")
resolved_user = os.getenv("VECDB_USERNAME")
resolved_password = os.getenv("VECDB_PASSWORD")
resolved_access_token = os.getenv("VECDB_ACCESS_TOKEN")

config_kwargs = {"rest_url": rest_url}
if resolved_access_token:
    config_kwargs["access_token"] = resolved_access_token
else:
    config_kwargs["username"] = resolved_user
    config_kwargs["password"] = resolved_password
config = Configuration(**config_kwargs)
# This can be used with local AI Database with ORDS using self signed certs
if os.getenv("VECDB_SELF_SIGNED_SSL", "false").lower() == "true":
    config.verify_ssl = False
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)



vecdb = OracleVecDB(config)
auth_method = 'bearer token' if config.access_token else 'username/password'
print(f"SDK client ready, using REST endpoint: {config.rest_url}")
print('Auth method:', auth_method)

## 4. Quick Health Check
Ping the service and inspect vector database statistics to confirm connectivity.

In [ ]:
db_stats = vecdb.describe_vector_database()
print(
    f"Models: {db_stats.total_models} | Tables: {db_stats.total_tables} | Vectors: {db_stats.total_vectors:,}"
)

## 5. Create or Reuse Parks Table
Create the `PARKS` table when it is not already present, then seed a small sample dataset for the browse and search examples.


In [ ]:
existing_tables = vecdb.list_vector_tables().items or []
table_exists = any(t.table_name == PARK_TABLE for t in existing_tables)

if table_exists:
    print(f"Using existing vector table: {PARK_TABLE}")
else:
    print(f"Creating vector table: {PARK_TABLE}")
    vecdb.create_vector_table(
        name=PARK_TABLE,
        comment="National parks demo table with integrated embeddings",
        table_params={"auto_generate_id": False},
        embed_params={
            "model": "all_MiniLM_L12_v2",
            "embed_metadata_jsonpath": "DESCRIPTION",
        },
        annotations={
            "STATES": "string",
            "NAME": "string",
            "COUNTRY_ID": "string",
            "LONGITUDE": "float",
            "LATITUDE": "float",
            "DESIGNATION": "string",
            "URL": "string",
            "PARK_CODE": "string",
            "WEATHER_INFO": "string",
            "DESCRIPTION": "string",
            "DIRECTIONS_INFO": "string",
            "DIRECTIONS_URL": "string",
            "FULL_NAME": "string",
        },
        index_params={
            "metadata_index_params": {
                "auto_index": True,
                "include_paths": ["STATES", "DESIGNATION", "PARK_CODE"],
            },
        },
    )

    sample_parks = [
        {
            "id": "PARK-YOSE-001",
            "metadata": {
                "STATES": "CA",
                "NAME": "Yosemite",
                "COUNTRY_ID": "USA",
                "LONGITUDE": -119.5383,
                "LATITUDE": 37.8651,
                "DESIGNATION": "National Park",
                "URL": "https://www.nps.gov/yose/",
                "PARK_CODE": "yose",
                "WEATHER_INFO": "Mountain weather varies by elevation and season.",
                "DESCRIPTION": "Yosemite protects granite cliffs, waterfalls, giant sequoias, wildlife habitat, and High Sierra wilderness.",
                "DIRECTIONS_INFO": "Accessible by road from western Sierra Nevada gateways.",
                "DIRECTIONS_URL": "https://www.nps.gov/yose/planyourvisit/directions.htm",
                "FULL_NAME": "Yosemite National Park",
            },
        },
        {
            "id": "PARK-JOTR-001",
            "metadata": {
                "STATES": "CA",
                "NAME": "Joshua Tree",
                "COUNTRY_ID": "USA",
                "LONGITUDE": -115.9007,
                "LATITUDE": 33.8734,
                "DESIGNATION": "National Park",
                "URL": "https://www.nps.gov/jotr/",
                "PARK_CODE": "jotr",
                "WEATHER_INFO": "Desert conditions with hot summers and cooler winters.",
                "DESCRIPTION": "Joshua Tree joins Mojave and Colorado Desert ecosystems with rugged rock formations, desert plants, historic sites, and night skies.",
                "DIRECTIONS_INFO": "Entrances are near Joshua Tree, Twentynine Palms, and Cottonwood Spring.",
                "DIRECTIONS_URL": "https://www.nps.gov/jotr/planyourvisit/directions.htm",
                "FULL_NAME": "Joshua Tree National Park",
            },
        },
        {
            "id": "PARK-YELL-001",
            "metadata": {
                "STATES": "WY,MT,ID",
                "NAME": "Yellowstone",
                "COUNTRY_ID": "USA",
                "LONGITUDE": -110.5885,
                "LATITUDE": 44.4280,
                "DESIGNATION": "National Park",
                "URL": "https://www.nps.gov/yell/",
                "PARK_CODE": "yell",
                "WEATHER_INFO": "High-elevation weather changes quickly throughout the year.",
                "DESCRIPTION": "Yellowstone features geysers, hot springs, canyons, lakes, forests, bison, wolves, and other wildlife.",
                "DIRECTIONS_INFO": "Five entrances connect the park to communities in Wyoming, Montana, and Idaho.",
                "DIRECTIONS_URL": "https://www.nps.gov/yell/planyourvisit/directions.htm",
                "FULL_NAME": "Yellowstone National Park",
            },
        },
        {
            "id": "PARK-MESA-001",
            "metadata": {
                "STATES": "CO",
                "NAME": "Mesa Verde",
                "COUNTRY_ID": "USA",
                "LONGITUDE": -108.4618,
                "LATITUDE": 37.2309,
                "DESIGNATION": "National Park",
                "URL": "https://www.nps.gov/meve/",
                "PARK_CODE": "meve",
                "WEATHER_INFO": "Four-season high desert climate.",
                "DESCRIPTION": "Mesa Verde preserves Ancestral Pueblo cliff dwellings, archeological sites, mesa-top trails, and cultural history.",
                "DIRECTIONS_INFO": "The park entrance is near Cortez and Mancos in southwestern Colorado.",
                "DIRECTIONS_URL": "https://www.nps.gov/meve/planyourvisit/directions.htm",
                "FULL_NAME": "Mesa Verde National Park",
            },
        },
    ]

    vecdb.upsert_vectors(table_name=PARK_TABLE, vectors=sample_parks)
    print(f"Seeded {len(sample_parks)} sample parks.")


## 6. Table Overview
Describe the parks vector table to examine index settings and annotations.

In [ ]:
from pprint import pprint

park_table = vecdb.describe_vector_table(name=PARK_TABLE)

table_snapshot = {
    "table_name": park_table.table_name,
    "vector_type": park_table.vector_type,
    "comment": getattr(park_table, "comment", None) or getattr(park_table, "description", None),
}

pprint(table_snapshot)

## 7. Browse Sample Parks
Preview a handful of park rows to understand available metadata fields.

In [ ]:
def query_items(response):
    if isinstance(response, list):
        return response
    if isinstance(response, tuple):
        return list(response)
    return getattr(response, "items", None) or getattr(response, "matches", None) or []


def result_metadata(item):
    return item.get("metadata", {}) if isinstance(item, dict) else getattr(item, "metadata", {})


def result_distance(item):
    if isinstance(item, dict):
        return item.get("distance")
    return getattr(item, "distance", getattr(item, "score", None))


def result_id(item):
    return item.get("id") if isinstance(item, dict) else getattr(item, "id", None)


def result_vector(item):
    if isinstance(item, dict):
        return item.get("vector") or item.get("dense_vector")
    return getattr(item, "vector", getattr(item, "dense_vector", None))


def result_text(item):
    return item.get("text", "") if isinstance(item, dict) else getattr(item, "text", "")


import pandas as pd
from textwrap import shorten

preview_results = vecdb.query(
    table_name=PARK_TABLE,
    query_by={"text": "national park"},
    top_k=N_RESULTS,
    include_vectors=False,
)

rows = [
    {
        "Name": (meta := result_metadata(item)).get("NAME"),
        "State": meta.get("STATES"),
        "URL": meta.get("URL"),
        "Description": meta.get("DESCRIPTION"),
        "Distance": round(result_distance(item) or 0.0, 4),
    }
    for item in query_items(preview_results)
]

pd.DataFrame(rows)

## 8. Filtered Semantic Search
Blend semantic similarity with metadata filters to find parks that match narrative context and specific criteria.

In [ ]:
def query_items(response):
    if isinstance(response, list):
        return response
    if isinstance(response, tuple):
        return list(response)
    return getattr(response, "items", None) or getattr(response, "matches", None) or []


def result_metadata(item):
    return item.get("metadata", {}) if isinstance(item, dict) else getattr(item, "metadata", {})


def result_distance(item):
    if isinstance(item, dict):
        return item.get("distance")
    return getattr(item, "distance", getattr(item, "score", None))


def result_id(item):
    return item.get("id") if isinstance(item, dict) else getattr(item, "id", None)


def result_vector(item):
    if isinstance(item, dict):
        return item.get("vector") or item.get("dense_vector")
    return getattr(item, "vector", getattr(item, "dense_vector", None))


def result_text(item):
    return item.get("text", "") if isinstance(item, dict) else getattr(item, "text", "")


search_text = "I want a park where I can learn some history and see animals"

filtered = vecdb.query(
    table_name=PARK_TABLE,
    query_by={"text": search_text},
    filters={"$and": [{"STATES": {"$eq": "CA"}}]},
    top_k=N_RESULTS,
    include_vectors=False,
)

pd.DataFrame(
    {
        "Name": result_metadata(item).get("NAME"),
        "State": result_metadata(item).get("STATES"),
        "Website": result_metadata(item).get("URL"),
        "Distance": round(result_distance(item) or 0.0, 4),
        "Description": shorten(result_metadata(item).get("DESCRIPTION", ""), width=120, placeholder="…"),
    }
    for item in query_items(filtered)
)

## 9. Generate Demo Embedding and Upsert
Create an embedding for the demo park description, then upsert a temporary park row using integrated table embeddings.

In [ ]:
demo_description = "Demo Ridge Park offers gentle hikes, bird watching, and stargazing."

embedding_response = vecdb.generate_embedding(
    model_name="all_MiniLM_L12_v2",
    inputs=[demo_description],
)
print("Generated embedding dimensions:", len(embedding_response.data[0].embedding))

demo_vector = {
    "id": "DEMO-RIDGE-001",
    "metadata": {
        "STATES": "OR",
        "NAME": "Demo Ridge Park",
        "COUNTRY_ID": "USA",
        "LONGITUDE": -122.486732,
        "LATITUDE": 45.345678,
        "DESIGNATION": "Scenic Recreation Area",
        "URL": "https://example.com/demo-ridge",
        "PARK_CODE": "demp",
        "WEATHER_INFO": "Mild coastal climate; light rain common in spring and fall.",
        "DESCRIPTION": demo_description,
        "DIRECTIONS_INFO": "Trailhead is 5 miles east of Demo Ridge town center on Hwy 26.",
        "DIRECTIONS_URL": "https://example.com/demo-ridge/directions",
        "FULL_NAME": "Demo Ridge National Scenic Area",
    },
}

upsert_response = vecdb.upsert_vectors(table_name=PARK_TABLE, vectors=[demo_vector])
print("Upsert status:", upsert_response)


## 10. Cleanup Demo
, then delete it to keep the data set tidy.


In [ ]:
vecdb.delete_vectors(table_name=PARK_TABLE, ids=[demo_vector["id"]])
print("Cleanup complete.")